# RAG Query Pipeline — Retrieval & Generation

Requires `bm25_params.json` and a populated Pinecone index from `rag_pipeline.ipynb`.

Re-run freely when tuning retrieval or prompt. For evaluation, use `rag_eval.ipynb`.

## Setup — Load credentials

In [ ]:
import os
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(PROJECT_ROOT)
print(f"Working directory: {os.getcwd()}")

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

PINECONE_API_KEY = os.environ["PINECONE_API_KEY"]
OPENAI_API_KEY = os.environ["OPENAI_API_KEY"]
ANTHROPIC_API_KEY = os.environ["ANTHROPIC_API_KEY"]

print("Credentials loaded.")

---
## Module 6 — Retriever Setup
Load BM25 from disk, connect to Pinecone, set up `PineconeHybridSearchRetriever`.

**Test:** Run a sample query, inspect top-5 results and source files.

In [ ]:
from pinecone_text.sparse import BM25Encoder
from langchain_openai import OpenAIEmbeddings
from langchain_community.retrievers import PineconeHybridSearchRetriever
from pinecone import Pinecone

bm25_loaded = BM25Encoder()
bm25_loaded.load("data/bm25_params.json")

embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small",
    openai_api_key=OPENAI_API_KEY,
)

pc = Pinecone(api_key=PINECONE_API_KEY)
index = pc.Index("langchain-core-rag")

retriever = PineconeHybridSearchRetriever(
    embeddings=embeddings,
    sparse_encoder=bm25_loaded,
    index=index,
    top_k=5,
    alpha=0.5,
    text_key="text",
)

results = retriever.invoke("how does RunnableSequence work?")
print(f"Retrieved {len(results)} chunks\n")
for r in results:
    print(f"Source : {r.metadata.get('source', 'unknown')}")
    print(f"Preview: {r.page_content[:150]}")
    print("---")

---
## Module 7 — RAG Chain + Generation
Wire retriever → prompt → Claude into a LangChain chain.

**Test:** Ask questions, verify answers use only retrieved context. Verify fallback for unknown questions.

In [ ]:
from langchain_anthropic import ChatAnthropic
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

llm = ChatAnthropic(
    model="claude-sonnet-4-6",
    anthropic_api_key=ANTHROPIC_API_KEY,
)

prompt = ChatPromptTemplate.from_template("""\
You are a code assistant for the LangChain library.
Answer strictly using only the retrieved context below — no outside knowledge.
If the context does not contain enough information, respond with:
\"The information needed to answer this question is not available in the provided context.\"
Do not infer, guess, or make up class names, method names, or code behaviour.

Context:
{context}

Question: {question}
""")

def format_docs(docs):
    return "\n\n".join(
        f"[{d.metadata.get('source', '')}]\n{d.page_content}" for d in docs
    )

chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

test_questions = [
    "What is the purpose of RunnableLambda?",
    "How does RunnableSequence chain multiple runnables?",
    "What does the ConfigurableField class do?",
]

for q in test_questions:
    print(f"Q: {q}")
    print(f"A: {chain.invoke(q)}")
    print("---")